# Doc-to-LoRA: Learning to Instantly Internalize Contexts

**Goal**: After this session, you will be able to implement a Perceiver-based hypernetwork that generates LoRA weights from documents, from memory.
**Time**: 30 minutes.

## What and Why
Doc-to-LoRA (D2L) meta-learns a hypernetwork that performs context distillation in a single forward pass. Given a document, it generates LoRA adapter weights for a frozen LLM, eliminating repeated context processing and reducing KV-cache memory. The key components: a Perceiver aggregator that compresses variable-length documents to fixed-shape latents, a projection head that maps latents to LoRA matrices, and a top-K distillation loss for training.

## Key Formulas

**Context distillation objective:**
$$\min_\phi \mathbb{E}_{(x,y) \sim \mathcal{D}_c} \left[ \text{KL}\left( p_\theta(y \mid x, c) \| p_{\theta + H_\phi(c)}(y \mid x) \right) \right]$$

**Perceiver cross-attention:**
$$U_l = \text{XAttn}(Q_m, K(Z_{l-1}), V(Z_{l-1})) \in \mathbb{R}^{r \times d_u}$$

**LoRA injection per layer:**
$$W'_l = W_l + \alpha_l B_l A_l; \quad A_l \in \mathbb{R}^{r \times d_l^{in}}, \quad B_l \in \mathbb{R}^{d_l^{out} \times r}$$

**Chunk-and-merge (longer contexts):**
$$A_l = \begin{bmatrix} A_l^{(1)} \\ \vdots \\ A_l^{(K)} \end{bmatrix}, \quad B_l = \begin{bmatrix} B_l^{(1)} \cdots B_l^{(K)} \end{bmatrix}$$

## References
- Paper: https://arxiv.org/abs/2602.15902
- Code: https://github.com/SakanaAI/doc-to-lora

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from jaxtyping import Float
from torch import Tensor
from einops import rearrange, repeat
import math

# --- Provided: LoRA forward (you've done this before) ---
def lora_forward(
    x: Float[Tensor, "batch seq d_in"],
    weight: Float[Tensor, "d_out d_in"],
    bias: Float[Tensor, "d_out"] | None,
    lora_A: Float[Tensor, "rank d_in"],
    lora_B: Float[Tensor, "rank d_out"],
    scaling: float = 1.0,
) -> Float[Tensor, "batch seq d_out"]:
    """Apply linear layer with externally-provided LoRA: y = xW^T + b + scaling * (x @ A^T) @ B"""
    base_out = F.linear(x, weight, bias)
    lora_down = torch.einsum("b s d, r d -> b s r", x, lora_A)
    lora_up = torch.einsum("b s r, r o -> b s o", lora_down, lora_B)
    return base_out + scaling * lora_up

# --- Provided: Chunk utilities ---
def chunk_and_merge_lora(A_chunks, B_chunks):
    """Merge chunk-wise LoRA by concatenating along rank dimension."""
    return torch.cat(A_chunks, dim=1), torch.cat(B_chunks, dim=1)

def chunk_document(token_features, chunk_size):
    """Split document features into fixed-size chunks with zero-padding."""
    total_seq, d = token_features.shape[1], token_features.shape[2]
    chunks = []
    for start in range(0, total_seq, chunk_size):
        end = min(start + chunk_size, total_seq)
        chunk = token_features[:, start:end, :]
        if chunk.shape[1] < chunk_size:
            pad = torch.zeros(1, chunk_size - chunk.shape[1], d, device=chunk.device, dtype=chunk.dtype)
            chunk = torch.cat([chunk, pad], dim=1)
        chunks.append(chunk)
    return chunks

## Part 1: Top-K Context Distillation Loss
The training loss for D2L. Instead of matching the full vocabulary distribution (expensive), match only the teacher's top-K token probabilities — a sparse approximation that captures the important mass.

<details><summary>Hint: numerical stability</summary>Use logsumexp for computing log-denominators instead of log(sum(exp(...)))</details>

In [ ]:
def topk_distillation_loss(
    teacher_logits: Float[Tensor, "batch vocab"],
    student_logits: Float[Tensor, "batch vocab"],
    K: int = 16,
) -> Float[Tensor, ""]:
    """Top-K cross-entropy: -sum_k p_teacher(k) * log q_student(k), averaged over batch."""
    # Get teacher's top-K logits and their indices
    topk_vals, topk_idx = teacher_logits.topk(K, dim=-1)  # (batch, K)

    # Teacher probabilities over top-K (logsumexp for numerical stability)
    teacher_log_denom = torch.logsumexp(teacher_logits.float(), dim=-1, keepdim=True)
    teacher_probs = (topk_vals.float() - teacher_log_denom).exp().detach()  # (batch, K)

    # Student log-probabilities at teacher's top-K positions
    student_selected = student_logits.gather(-1, topk_idx)  # (batch, K)
    student_log_denom = torch.logsumexp(student_logits.float(), dim=-1, keepdim=True)
    student_log_probs = student_selected.float() - student_log_denom  # (batch, K)

    # Cross-entropy
    token_losses = -(teacher_probs * student_log_probs).sum(dim=-1)  # (batch,)
    return token_losses.mean()

In [ ]:
# --- Part 1 Validation ---
torch.manual_seed(42)
batch_size, vocab_size, K = 8, 1000, 16
teacher_logits = torch.randn(batch_size, vocab_size)
student_logits = torch.randn(batch_size, vocab_size)

loss = topk_distillation_loss(teacher_logits, student_logits, K=K)

# Shape
assert loss.shape == (), f"Shape: expected scalar, got {loss.shape}"
print(f"  Shape: scalar -- correct")

# Diagnostics
print(f"  Loss value: {loss.item():.4f}")

# Non-negative (cross-entropy)
assert loss >= 0, f"Cross-entropy loss must be non-negative, got {loss:.4f}"
print(f"  Non-negative -- correct")

# Self-distillation should be lower than random
loss_self = topk_distillation_loss(teacher_logits, teacher_logits, K=K)
assert loss_self < loss, f"Self-distill ({loss_self:.4f}) should be < random ({loss:.4f})"
print(f"  Self-distill ({loss_self:.4f}) < random ({loss:.4f}) -- correct")

# Reference match
topk_vals, topk_idx = teacher_logits.topk(K, dim=-1)
t_log_denom = torch.logsumexp(teacher_logits.float(), dim=-1, keepdim=True)
t_probs = (topk_vals.float() - t_log_denom).exp()
s_selected = student_logits.gather(-1, topk_idx)
s_log_denom = torch.logsumexp(student_logits.float(), dim=-1, keepdim=True)
s_log_probs = s_selected.float() - s_log_denom
ref_loss = -(t_probs * s_log_probs).sum(dim=-1).mean()
max_diff = (loss - ref_loss).abs()
assert torch.allclose(loss, ref_loss, atol=1e-5), f"Max diff: {max_diff:.2e} (threshold: 1e-5)"
print(f"  Reference match -- correct (diff: {max_diff:.2e})")

# Gradient flow
s_grad = student_logits.clone().requires_grad_(True)
topk_distillation_loss(teacher_logits, s_grad, K=K).backward()
assert s_grad.grad is not None and not torch.all(s_grad.grad == 0)
print(f"  Gradient flow -- correct")

print("\nPart 1 complete.")

## Part 2: Perceiver Cross-Attention Aggregator
The core of D2L's architecture. Maps variable-length document token features to a fixed number of latent vectors via cross-attention (latent queries attend to input tokens), followed by self-attention refinement. Output shape is always `(batch, num_latents, d_latent)` regardless of input length — this is how D2L handles arbitrary document sizes.

In [ ]:
class PerceiverAggregator(nn.Module):
    """Perceiver cross-attention: variable-length input -> fixed-shape latent output."""

    def __init__(
        self,
        d_input: int,
        d_latent: int,
        num_latents: int,
        num_heads: int = 4,
        num_self_attn_blocks: int = 1,
    ):
        super().__init__()
        self.d_latent = d_latent
        self.num_latents = num_latents
        self.num_heads = num_heads

        # Learned latent queries
        self.latent_queries = nn.Parameter(torch.randn(num_latents, d_latent) * 0.02)

        # Input projection
        self.input_proj = nn.Linear(d_input, d_latent)

        # Cross-attention
        self.cross_attn = nn.MultiheadAttention(embed_dim=d_latent, num_heads=num_heads, batch_first=True)
        self.cross_norm_q = nn.LayerNorm(d_latent)
        self.cross_norm_kv = nn.LayerNorm(d_latent)

        # Self-attention blocks
        self.self_attn_blocks = nn.ModuleList()
        for _ in range(num_self_attn_blocks):
            self.self_attn_blocks.append(nn.ModuleDict({
                "attn": nn.MultiheadAttention(embed_dim=d_latent, num_heads=num_heads, batch_first=True),
                "norm1": nn.LayerNorm(d_latent),
                "ffn": nn.Sequential(nn.Linear(d_latent, d_latent * 4), nn.SiLU(), nn.Linear(d_latent * 4, d_latent)),
                "norm2": nn.LayerNorm(d_latent),
            }))

    def forward(
        self,
        token_features: Float[Tensor, "batch seq_len d_input"],
        attention_mask: Float[Tensor, "batch seq_len"] | None = None,
    ) -> Float[Tensor, "batch num_latents d_latent"]:
        """Cross-attend latent queries to input tokens, then refine with self-attention."""
        batch_size = token_features.shape[0]

        # Project input
        kv = self.input_proj(token_features)

        # Expand latent queries
        queries = repeat(self.latent_queries, "n d -> b n d", b=batch_size)

        # Attention mask
        key_padding_mask = None
        if attention_mask is not None:
            key_padding_mask = (attention_mask == 0)

        # Cross-attention
        q_normed = self.cross_norm_q(queries)
        kv_normed = self.cross_norm_kv(kv)
        cross_out, _ = self.cross_attn(query=q_normed, key=kv_normed, value=kv_normed, key_padding_mask=key_padding_mask)
        latents = queries + cross_out

        # Self-attention refinement
        for block in self.self_attn_blocks:
            normed = block["norm1"](latents)
            sa_out, _ = block["attn"](query=normed, key=normed, value=normed)
            latents = latents + sa_out

            normed = block["norm2"](latents)
            ffn_out = block["ffn"](normed)
            latents = latents + ffn_out

        return latents

In [ ]:
# --- Part 2 Validation ---
torch.manual_seed(42)
batch, d_input, d_latent, num_latents, num_heads = 2, 256, 128, 8, 4

perceiver = PerceiverAggregator(
    d_input=d_input, d_latent=d_latent, num_latents=num_latents,
    num_heads=num_heads, num_self_attn_blocks=2,
)

# Shape: fixed output regardless of input length
for seq_len in [32, 64, 128, 256]:
    tokens = torch.randn(batch, seq_len, d_input)
    out = perceiver(tokens)
    assert out.shape == (batch, num_latents, d_latent), f"seq_len={seq_len}: expected {(batch, num_latents, d_latent)}, got {out.shape}"
print(f"  Shape: (batch, {num_latents}, {d_latent}) for all input lengths -- correct")

# Diagnostics
out = perceiver(torch.randn(batch, 64, d_input))
print(f"  Range: [{out.min():.4f}, {out.max():.4f}]")
print(f"  Mean:  {out.mean():.4f}, Std: {out.std():.4f}")

# Attention mask changes output
tokens = torch.randn(batch, 64, d_input)
mask_full = torch.ones(batch, 64)
mask_half = torch.ones(batch, 64)
mask_half[:, 32:] = 0
out_full = perceiver(tokens, attention_mask=mask_full)
out_half = perceiver(tokens, attention_mask=mask_half)
assert not torch.allclose(out_full, out_half, atol=1e-4), "Masking must change output"
print(f"  Attention mask sensitivity -- correct")

# Gradient flow
tokens_g = torch.randn(batch, 64, d_input, requires_grad=True)
perceiver(tokens_g).sum().backward()
assert tokens_g.grad is not None and not torch.all(tokens_g.grad == 0)
print(f"  Gradient flow -- correct")

print("\nPart 2 complete.")

## Part 3: HyperLoRA Head
Takes the Perceiver's latent vectors and projects them into LoRA A and B matrices for each target layer. Each latent vector becomes one rank of the adapter. Includes residual MLP processing, per-layer projection weights, and learnable bias/scaling — B is zero-initialized (like standard LoRA) so the hypernetwork starts as identity.

In [ ]:
class ResMLPBlock(nn.Module):
    """Residual MLP: x + MLP(LayerNorm(x))"""
    def __init__(self, d_model: int, d_hidden: int):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, d_hidden),
            nn.SiLU(), nn.Linear(d_hidden, d_model), nn.LayerNorm(d_model),
        )
    def forward(self, x):
        return x + self.mlp(x)


class HyperLoRAHead(nn.Module):
    """Project perceiver latents -> per-layer LoRA A, B matrices."""

    def __init__(
        self,
        d_latent: int,
        d_in: int,
        d_out: int,
        n_layers: int,
        rank: int,
        num_mlp_blocks: int = 2,
    ):
        super().__init__()
        self.d_in = d_in
        self.d_out = d_out
        self.n_layers = n_layers
        self.rank = rank

        # Residual MLP blocks
        self.layers = nn.Sequential(*[ResMLPBlock(d_latent, d_latent * 4) for _ in range(num_mlp_blocks)])

        # Per-layer projection weight
        self.head_weight = nn.Parameter(
            torch.randn(n_layers, d_latent, d_in + d_out) * (0.5 / math.sqrt(d_latent + (d_in + d_out) * rank))
        )

        # Learnable bias
        self.bias_A = nn.Parameter(torch.randn(n_layers, rank, d_in) * (0.2 / math.sqrt(d_in * rank)))
        self.bias_B = nn.Parameter(torch.zeros(n_layers, rank, d_out))

        # Learnable scaling
        self.scaler_A = nn.Parameter(torch.ones(1, n_layers, rank, 1))
        self.scaler_B = nn.Parameter(torch.zeros(1, n_layers, rank, 1))

    def forward(
        self,
        latents: Float[Tensor, "batch n_layers rank d_latent"],
    ) -> tuple[
        Float[Tensor, "batch n_layers rank d_in"],
        Float[Tensor, "batch n_layers rank d_out"],
    ]:
        """Generate LoRA matrices: MLP -> normalize -> project -> scale + bias."""
        # Process through MLP
        h = self.layers(latents)

        # Normalize
        h = h / (h.norm(dim=-1, keepdim=True) + 1e-8)

        # Project to (d_in + d_out) per layer
        flat_lora = torch.einsum("b l r d, l d o -> b l r o", h, self.head_weight)

        # Split into A and B
        A_raw, B_raw = flat_lora.split([self.d_in, self.d_out], dim=-1)

        # Scale and bias
        A = A_raw * self.scaler_A + self.bias_A.unsqueeze(0)
        B = B_raw * self.scaler_B + self.bias_B.unsqueeze(0)

        return A, B

In [ ]:
# --- Part 3 Validation ---
torch.manual_seed(42)
batch, d_latent, d_in, d_out, n_layers, rank = 2, 128, 256, 512, 4, 8

head = HyperLoRAHead(d_latent=d_latent, d_in=d_in, d_out=d_out, n_layers=n_layers, rank=rank)
latents = torch.randn(batch, n_layers, rank, d_latent)
A, B = head(latents)

# Shape
assert A.shape == (batch, n_layers, rank, d_in), f"A shape: {A.shape}"
assert B.shape == (batch, n_layers, rank, d_out), f"B shape: {B.shape}"
print(f"  Shape: A={A.shape}, B={B.shape} -- correct")

# Diagnostics
print(f"  A range: [{A.min():.4f}, {A.max():.4f}], mean: {A.mean():.4f}")
print(f"  B range: [{B.min():.6f}, {B.max():.6f}], mean: {B.mean():.6f}")

# B should be ~0 at init (scaler_B=0, bias_B=0)
assert B.abs().max() < 1e-5, f"B should be ~0 at init, got max={B.abs().max():.2e}"
print(f"  Zero-init B (like standard LoRA) -- correct")

# A should be non-zero (from bias_A)
assert A.abs().max() > 1e-6, "A should be non-zero from bias_A"
print(f"  Non-zero A init -- correct")

# Different inputs -> different A
A2, _ = head(torch.randn(batch, n_layers, rank, d_latent))
assert not torch.allclose(A, A2, atol=1e-5)
print(f"  Input sensitivity -- correct")

# Gradient flow
lat_g = torch.randn(batch, n_layers, rank, d_latent, requires_grad=True)
A_g, B_g = head(lat_g)
(A_g.sum() + B_g.sum()).backward()
assert lat_g.grad is not None and not torch.all(lat_g.grad == 0)
print(f"  Gradient flow -- correct")

# Integration: generated LoRA works with lora_forward
x_test = torch.randn(1, 4, d_in)
W_test = torch.randn(d_out, d_in)
out = lora_forward(x_test, W_test, None, A[0, 0], B[0, 0], scaling=1.0)
assert out.shape == (1, 4, d_out)
print(f"  Integration with lora_forward -- correct")

print("\nPart 3 complete.")

## Part 4: End-to-End Doc-to-LoRA Pipeline
Wire everything together: document features -> (optional chunking) -> Perceiver -> layer expansion -> HyperLoRA head -> (optional merge) -> LoRA A, B matrices. Then demonstrate context distillation training.

In [ ]:
class DocToLoRA(nn.Module):
    """Complete Doc-to-LoRA hypernetwork."""

    def __init__(self, d_input, d_latent, d_model_in, d_model_out,
                 n_layers, rank, num_heads=4, num_self_attn_blocks=1,
                 num_mlp_blocks=2, chunk_size=None):
        super().__init__()
        self.n_layers = n_layers
        self.rank = rank
        self.chunk_size = chunk_size

        self.perceiver = PerceiverAggregator(
            d_input=d_input, d_latent=d_latent, num_latents=rank,
            num_heads=num_heads, num_self_attn_blocks=num_self_attn_blocks,
        )
        self.hyper_head = HyperLoRAHead(
            d_latent=d_latent, d_in=d_model_in, d_out=d_model_out,
            n_layers=n_layers, rank=rank, num_mlp_blocks=num_mlp_blocks,
        )
        self.layer_expand = nn.Linear(d_latent, d_latent * n_layers)

    def _process_single_chunk(self, chunk_features, attention_mask=None):
        """Process one chunk: perceiver -> expand across layers -> hyper_head."""
        latents = self.perceiver(chunk_features, attention_mask)
        expanded = self.layer_expand(latents)
        expanded = rearrange(expanded, "b r (l d) -> b l r d", l=self.n_layers)
        A, B = self.hyper_head(expanded)
        return A.squeeze(0), B.squeeze(0)

    def forward(self, doc_features, attention_mask=None):
        """Generate LoRA adapters, with optional chunking for long documents."""
        if self.chunk_size is None or doc_features.shape[1] <= self.chunk_size:
            return self._process_single_chunk(doc_features, attention_mask)

        chunks = chunk_document(doc_features, self.chunk_size)
        total_seq = doc_features.shape[1]
        A_list, B_list = [], []

        for i, chunk in enumerate(chunks):
            start = i * self.chunk_size
            end = min(start + self.chunk_size, total_seq)
            real_len = end - start
            chunk_mask = torch.zeros(1, self.chunk_size, device=doc_features.device)
            chunk_mask[:, :real_len] = 1.0
            A_i, B_i = self._process_single_chunk(chunk, chunk_mask)
            A_list.append(A_i)
            B_list.append(B_i)

        return chunk_and_merge_lora(A_list, B_list)

In [ ]:
# --- Part 4 Validation ---
torch.manual_seed(42)

d_input, d_latent, d_model_in, d_model_out = 128, 64, 128, 128
n_layers, rank, chunk_size = 4, 4, 32

d2l = DocToLoRA(
    d_input=d_input, d_latent=d_latent, d_model_in=d_model_in,
    d_model_out=d_model_out, n_layers=n_layers, rank=rank, chunk_size=chunk_size,
)

# Short document (no chunking)
short_doc = torch.randn(1, 20, d_input)
A, B = d2l(short_doc)
assert A.shape == (n_layers, rank, d_model_in), f"Short doc A: {A.shape}"
assert B.shape == (n_layers, rank, d_model_out), f"Short doc B: {B.shape}"
print(f"  Short doc: A={A.shape}, B={B.shape} -- correct")

# Long document (with chunking)
long_doc = torch.randn(1, 100, d_input)
A_long, B_long = d2l(long_doc)
n_chunks = math.ceil(100 / chunk_size)
assert A_long.shape == (n_layers, n_chunks * rank, d_model_in)
print(f"  Long doc: {n_chunks} chunks x rank {rank} = merged rank {n_chunks * rank} -- correct")

# Diagnostics
print(f"  A range: [{A.min():.4f}, {A.max():.4f}]")
print(f"  B range: [{B.min():.6f}, {B.max():.6f}]")

# LoRA injection works
target_W = torch.randn(d_model_out, d_model_in)
x_in = torch.randn(1, 8, d_model_in)
for l in range(n_layers):
    out = lora_forward(x_in, target_W, None, A[l], B[l], scaling=1.0)
    assert out.shape == (1, 8, d_model_out)
print(f"  LoRA injection into target model -- correct")

# Training simulation
class TinyModel(nn.Module):
    def __init__(self, d, n_layers, vocab):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(d, d) for _ in range(n_layers)])
        self.head = nn.Linear(d, vocab)
    def forward(self, x, lora_As=None, lora_Bs=None):
        for i, layer in enumerate(self.layers):
            x = lora_forward(x, layer.weight, layer.bias, lora_As[i], lora_Bs[i]) if lora_As is not None else layer(x)
            x = F.silu(x)
        return self.head(x)

vocab = 100
model = TinyModel(d_model_in, n_layers, vocab)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

optimizer = torch.optim.Adam(d2l.parameters(), lr=1e-3)
doc = torch.randn(1, 40, d_input)
query = torch.randn(1, 8, d_model_in)

with torch.no_grad():
    teacher_logits = model(query) + 0.5 * torch.randn(1, 8, vocab)

losses = []
for step in range(50):
    optimizer.zero_grad()
    A, B = d2l(doc)
    student_logits = model(query, lora_As=A, lora_Bs=B)
    loss = topk_distillation_loss(teacher_logits.view(-1, vocab), student_logits.view(-1, vocab), K=16)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

assert losses[-1] < losses[0], f"Loss should decrease: {losses[0]:.4f} -> {losses[-1]:.4f}"
print(f"  Training: loss {losses[0]:.4f} -> {losses[-1]:.4f} -- correct")

params = sum(p.numel() for p in d2l.parameters())
print(f"  Hypernetwork params: {params:,}")

print("\nPart 4 complete.")

## Session Debrief

Without scrolling up, answer in your head:
1. What is the core formula for the context distillation objective?
2. How does the Perceiver handle variable-length inputs to produce fixed-shape output?
3. Why is B zero-initialized in the HyperLoRA head?

**Challenge**: Close this notebook, open a blank one, and rewrite the PerceiverAggregator from scratch without looking back.